# M03A 資料擷取實驗：下載流程與請求間隔驗證

本筆記紀錄 TDCS M03A 資料擷取流程的研究與實作，  
包含官方下載結構、動態日期網址、檔案儲存、HTTPS 憑證問題，以及長時間連續下載實驗。  

### 目標是 M03A資料集。
網址：https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/

會依序記錄我遇到的困難與挑戰，還有怎麼解決問題，若有遇到一些不懂的技術問題也都會放上來。  
(當然若太深入就不在此談，有興趣再另外查找)


## 1.擷取資料的40秒限制：
注意到網站中有這一句話：「重複擷取資料週期間距應大於40秒，以免影響他人下載權利。若發現有違反情事，本局得逕行中斷其連線，使用者不得異議。」  

但是明明我在短時間內人工點擊數次(不管是同一個檔案還是不同的檔案) 都可以正常下載，也沒有被中斷。因此我懷疑此話的真實性。雖然遵守規範很重要，但若這網路上寫的「規範」只不過是「沒有任何強制性的純文字」呢？   

若每次都要有40秒的限制，要實現爬蟲下載一段時間的資料，實際上是非常耗時間的。我認為應該先挑戰這個問題。若可以破解，可以大幅降低擷取資料的時間，若技術上不需要逐次等待 40 秒，可以大幅降低下載流程的實際執行時間，但請求數量仍隨日期數量線性成長。

因此我將進行驗證：

在該網站中進行以下操作：
1. 右鍵「檢查」(或F12)。  
2. 點選「Network」(網路)。    
3. 在網站中，在短時間內隨意點選數個檔案下載，並檢視相關資訊。
4. 詳細一點可以點選相應的tar.gz的「Header」檢視該筆請求（Request）的詳細細節。  

個人測試10秒內點選了10個檔案進行下載。得到的結果如下：

1. Status (HTTP回應狀態碼) 均為 200 (請求成功，伺服器已完成處理)  
2. Type (類型) 均為 document：代表這只是傳統放在伺服器目錄下的 .tar.gz 檔案。後端根本沒有經過一層「計數器」或「Session 驗證門閥」來判斷你上一次下載是什麼時候。  
3. 沒有後端 Rate Limit 阻擋：伺服器完全放行。  

如果後端有設 40 秒限制，第二個之後的 Request 在送出的那一瞬間就會被退回 429 Too Many Requests才對。  

因此我大膽判斷這個40秒限制高機率是網站寫的「沒有任何強制性的純文字」！  
以下便開始挑戰這個假說。  

當平台方選擇「只貼告示、不寫程式」時，基本上就等於放棄了技術層面的強制執行力。  

## 2.開始研究爬蟲程式：  
gemini 的測試建議：

```py
import time
import random
import requests
from datetime import datetime, timedelta

# 1. 關閉 SSL 驗證警告訊息 (可加可不加，加上去畫面比較乾淨)
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 設定起始與終點日期 (就用三天來測試)
start_date = datetime(2026, 1, 26)
end_date = datetime(2026, 1, 28)

base_url = "https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/M03A_{}.tar.gz"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime("%Y%m%d")
    file_url = base_url.format(date_str)
    
    try:
        # 2. 關鍵修正：加上 verify=False 跳過 SSL 憑證檢查
        res = requests.get(file_url, headers=headers, verify=False, timeout=15)
        
        if res.status_code == 200:
            with open(f"M03A_{date_str}.tar.gz", "wb") as f:
                f.write(res.content)
            print(f"✅ 成功下載：M03A_{date_str}.tar.gz")
        else:
            print(f"❌ 下載失敗 {date_str}，狀態碼：{res.status_code}")
            
    except Exception as e:
        print(f"⚠️ 連線發生例外狀況: {e}")
    
    # 增加微小亂數間隔
    time.sleep(random.uniform(1.0, 2.5))
    current_date += timedelta(days=1)
```

測試都沒問題，以下是細節解析，當作我個人的學習，稍微理解後再進行完整的爬蟲程式的修正。  


## 技術細節：

### 1.headers = ... 是做什麼?

```py
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
```
### 說明：
User-Agent（使用者代理）是 HTTP 請求頭（Request Header）裡的一行文字，用來告訴伺服器：「我是什麼作業系統、什麼類型的瀏覽器」。  

如果完全不寫 headers，Python 的 requests 套件在發起連線時，預設帶給伺服器的 User-Agent 會是：python-requests/2.31.0  

伺服器看到這行文字，秒懂「這是程式/爬蟲，不是人在用 Chrome 看網頁」，許多網站會直接拒絕連線。
實際上檢驗「Header」該筆請求（Request）的詳細細節中確實可以看到 ：
```
user-agent Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36
```

### 2.timeout=15 是？
```py
res = requests.get(file_url, headers=headers, verify=False, timeout=15)
```
### 說明：
timeout=15 並非限制整次下載只能花 15 秒，而是當連線或資料讀取在指定時間內沒有進展時，拋出 Timeout 例外。


## 引用的套件：datetime & timedelta 究竟是？

對於一個陌生又全新的套件，可以這樣做：
### 1. 可執行以下觀察：
```py
from datetime import datetime, timedelta

# 1. 想知道 timedelta 到底有哪些屬性與方法？
print(dir(timedelta))

# 2. 想知道 datetime.now() 的具體用法與參數？
help(datetime.now)
```

### 2. 看官方文件，抓出核心概念（Concepts）
如果搜尋 python datetime doc 進入官方文件，會發現它不是雜亂無章的，而是把模組拆成幾個核心物件：

| 物件／類別 | 簡單來說它代表什麼？ | 常見用途範例 |
|---|---|---|
| `date` | 只有日期（年、月、日） | `2026-07-25` |
| `time` | 只有時間（時、分、秒、微秒） | `14:30:00` |
| `datetime` | 日期＋時間（兩者合一） | `2026-07-25 14:30:00` |
| `timedelta` | 時間差（時間長度） | `3 天、2 小時、50 分鐘` |


In [ ]:
from datetime import datetime, timedelta

# 1. 想知道 timedelta 到底有哪些屬性與方法？
print(dir(timedelta))

# 2. 想知道 datetime.now() 的具體用法與參數？
help(datetime.now)

# 有興趣在另外查詢資訊，內容也許多有料的，我就不特別放上來。

['__abs__', '__add__', '__bool__', '__class__', '__delattr__', '__dir__', '__divmod__', '__doc__', '__eq__', '__floordiv__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__mod__', '__mul__', '__ne__', '__neg__', '__new__', '__pos__', '__radd__', '__rdivmod__', '__reduce__', '__reduce_ex__', '__repr__', '__rfloordiv__', '__rmod__', '__rmul__', '__rsub__', '__rtruediv__', '__setattr__', '__sizeof__', '__str__', '__sub__', '__subclasshook__', '__truediv__', 'days', 'max', 'microseconds', 'min', 'resolution', 'seconds', 'total_seconds']
Help on built-in function now:

now(tz=None) class method of datetime.datetime
    Returns new datetime object representing current time local to tz.

      tz
        Timezone object.

    If no tz is specified, uses local timezone.



In [3]:
## 測試
from datetime import datetime, timedelta

# 1. 取得當前時間點 (datetime)
now = datetime.now()
print("現在時間：", now)

# 2. 定義一個 7 天的時間長度 (timedelta)
seven_days = timedelta(days=7)

# 3. 數學運算：算出 7 天後的時間點
future = now + seven_days
print("七天後：", future)

# 4. 兩時間點相減，得到 timedelta
past = datetime(2026, 1, 1)
diff = now - past
print("今年已經過了：", diff.days, "天")

現在時間： 2026-07-25 23:50:05.238038
七天後： 2026-08-01 23:50:05.238038
今年已經過了： 205 天


## time 套件 與 datetime 套件 的關鍵差異？

### 1. 抽象層級完全不同。
| 比較維度 | `time` 模組 | `datetime` 模組 |
|---|---|---|
| 底層哲學 | 面向 C 語言與作業系統 | 面向物件（Object-Oriented） |
| 時間表達方式 | Timestamp（時間戳）或 Tuple | 物件（有 `.year`、`.month`、`.hour` 等屬性） |
| 主要用途 | 效能測量、簡單暫停（`time.sleep()`）、系統層級時間 | 日期時間運算、格式化輸出、跨時區處理、商業邏輯 |


### 2. 計算特性的差異
time 模組： 做時間計算時，幾乎只能對「秒數 (float)」做加減，例如 time.time() + 86400（86400 秒 = 1 天）。這很容易寫錯，而且遇到閏年、跨月時手算會非常痛苦。

datetime 模組： 利用 timedelta 封裝好了進位邏輯，你直接寫 days=30 或 weeks=2，它會自動幫你處理大小月與閏年。

只是想讓程式停頓一下（time.sleep(1)）或測量程式執行了幾秒 ➔ 用 time。  
只要涉及日曆、加減幾天/幾小時、格式化字串（strftime） ➔ 一律用 datetime。  

###  如何使得網址成為動態生成的模板？

關鍵在於這兩行：

```py
date_str = current_date.strftime("%Y%m%d")
file_url = base_url.format(date_str)
```
第一行：date_str = current_date.strftime("%Y%m%d")
作用：將日期物件轉為符合網址格式的字串。
運作機制：  
current_date 是一個 Python 的 datetime 日期物件（例如 2026 年 1 月 26 日）。  
strftime 的全名是 String Format Time（格式化時間字串）。  
%Y：四位數年份 (2026)  
%m：兩位數月份 (01)  
%d：兩位數日期 (26)  
執行後，date_str 就會變成純文字 "20260126"。  

第二行：file_url = base_url.format(date_str)  
作用：將動態字串填入網址模板的挖空處。  
運作機制：  
在程式碼的前面，我們定義了一個 base_url：  
base_url = "[https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/M03A](https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/M03A)_{}.tar.gz"  
注意網址裡的 {}，這在 Python 的字串處理中叫做預留位置（Placeholder）。  
當呼叫 .format(date_str) 時，Python 就會把 date_str（也就是 "20260126"）精準地塞進那個 {} 裡面。  
組合後的結果：  
[https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/M03A_20260126.tar.gz](https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/M03A_20260126.tar.gz)


### 另一種現代寫法：f-string（字串插值）

Python 3.6 之後引入了更直覺的 f-string 寫法，你甚至可以把兩行縮編成一行，效果完全一樣：
```py
# 直接在字串前加上 f，並把變數或格式化公式直接塞進 {} 裡面
file_url = f"https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/M03A_{current_date.strftime('%Y%m%d')}.tar.gz"
```


## 3. 根據我的需求修正爬蟲程式：

利用上面的自己完成看看，其實也只要改起始與結束日期即可。
但我想加入：下載到特定資料夾。

因此引入 Path 套件。

```py
output_dir = Path("../data/raw/M03A")
output_dir.mkdir(parents=True, exist_ok=True)
```
意思是：
Path(...)：建立路徑物件  
mkdir()：建立資料夾  
parents=True：上層資料夾不存在也一起建立  
exist_ok=True：資料夾已存在也不會報錯  

```py
file_path = output_dir / f"M03A_{date_str}.tar.gz"

with file_path.open("wb") as f:
    f.write(res.content)
```
/ 在 Path 裡不是除法，而是組合路徑：
../data/raw/M03A/archives + M03A_20260101.tar.gz    
得到：  ../data/raw/M03A/archives/M03A_20260101.tar.gz  



In [9]:
from pathlib import Path
from datetime import datetime, timedelta
import requests
import time
import random

output_dir = Path("../data/raw/M03A")
output_dir.mkdir(parents=True, exist_ok=True)

start_date = datetime(2026, 1, 1)
end_date = datetime(2026, 1, 31)

base_url = ("https://tisvcloud.freeway.gov.tw/history/TDCS/M03A/M03A_{}.tar.gz")

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

current_date = start_date
start = time.time()

while current_date <= end_date:
    date_str = current_date.strftime("%Y%m%d")
    file_url = base_url.format(date_str)

    res = requests.get(file_url,headers=headers,verify=False,timeout=15)

    if res.status_code == 200:
        file_path = output_dir / f"M03A_{date_str}.tar.gz"

        with file_path.open("wb") as f:
            f.write(res.content)

        print(f"下載成功：{file_path}")
    else:
        print(
            f"下載失敗：{date_str}，"
            f"狀態碼：{res.status_code}"
        )

    current_date += timedelta(days=1)

    if current_date <= end_date:
        time.sleep(random.uniform(1, 2.5))

end = time.time()

total_time = end - start
days = (end_date - start_date).days + 1

print(f"總耗時：{total_time:.2f} 秒")
print(f"平均每個檔案：{total_time / days:.2f} 秒")

下載成功：..\data\raw\M03A\M03A_20260101.tar.gz
下載成功：..\data\raw\M03A\M03A_20260102.tar.gz
下載成功：..\data\raw\M03A\M03A_20260103.tar.gz
下載成功：..\data\raw\M03A\M03A_20260104.tar.gz
下載成功：..\data\raw\M03A\M03A_20260105.tar.gz
下載成功：..\data\raw\M03A\M03A_20260106.tar.gz
下載成功：..\data\raw\M03A\M03A_20260107.tar.gz
下載成功：..\data\raw\M03A\M03A_20260108.tar.gz
下載成功：..\data\raw\M03A\M03A_20260109.tar.gz
下載成功：..\data\raw\M03A\M03A_20260110.tar.gz
下載成功：..\data\raw\M03A\M03A_20260111.tar.gz
下載成功：..\data\raw\M03A\M03A_20260112.tar.gz
下載成功：..\data\raw\M03A\M03A_20260113.tar.gz
下載成功：..\data\raw\M03A\M03A_20260114.tar.gz
下載成功：..\data\raw\M03A\M03A_20260115.tar.gz
下載成功：..\data\raw\M03A\M03A_20260116.tar.gz
下載成功：..\data\raw\M03A\M03A_20260117.tar.gz
下載成功：..\data\raw\M03A\M03A_20260118.tar.gz
下載成功：..\data\raw\M03A\M03A_20260119.tar.gz
下載成功：..\data\raw\M03A\M03A_20260120.tar.gz
下載成功：..\data\raw\M03A\M03A_20260121.tar.gz
下載成功：..\data\raw\M03A\M03A_20260122.tar.gz
下載成功：..\data\raw\M03A\M03A_20260123.tar.gz
下載成功：..\dat

## 4. 擴大時間範圍，做最後的驗證

上述程式抓一個月的時間都沒問題，接下來寫較為正式的py檔案，測試長時間的可能性！(還請看 download_m03a.py)
就從2026/1/1 到 2026/5/31 為期完整五個月來做測試好了。

執行成功，總耗時約334秒，網站完全沒有檔。
可下結論：在本次實驗環境、IP 與請求規模下，以 1～2.5 秒間隔連續下載 151 個 M03A 每日檔案，未觀察到伺服器立即執行 40 秒的技術性阻擋，亦未發生 403、429 或連線中斷。

照這套作法，即可在短時間內，實現自動化抓取資料。

心得：批判性思維：勇於質疑表面上的文字，想想是否有更好的做法與如何實踐。而不要被表面上的文字綁架。

